# 3a. BE vs SC Grid-Search Cross-Validation

**Purpose**: Analyse results from the manuscript's grid-search CV procedure,
classifying each animal as BE or SC.

**Protocol** (matching manuscript):
1. Expert sessions: accuracy ≥ 70% AND last 50% of Uniform sessions
2. Grid search: 4 shared params (σ_noise × A_repulsion) × 2 model-specific → ~8,000 combos
3. 2-fold CV: split by blocks (sessions), fit on train, evaluate update matrix MSE on test
4. 64 seeds → 64 avg test errors per model per animal
5. One-way ANOVA to compare BE vs SC per animal

**Data flow**: Cluster pickles (`cv_{ANIMAL}_seed{NNN}.pkl`) → load → ANOVA → plots → parameter report

---

**Sections**:
1. Configuration
2. Load cluster results
3. ANOVA model comparison
4. Per-animal violin + scatter plots
5. Winner summary
6. Best-fit parameters
7. Empirical vs model update matrices
8. Parameter distributions across seeds
9. Save outputs

## 1. Setup & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
from analysis.cv_utils import (
    load_cv_pickles, summarise_loaded_results,
    build_long_df, run_anova, build_summary_table,
    get_best_seed_params, format_params, extract_param_df,
    select_expert_sessions, sessions_to_old_df,
    compute_empirical_um, simulate_model_um, compute_matrix_error,
)
from plotting.cv import (
    plot_cv_comparison, plot_winner_summary,
    plot_um_comparison, plot_update_matrix,
    plot_param_distributions,
)
from behav_utils.data.loading import load_experiment

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit these paths
# ═══════════════════════════════════════════════════════════════════════════════

CONFIG_PATH = '../config.yaml'                    # behav_utils config
CV_RESULTS_DIR = '../results/cv'                  # directory with cv_*_seed*.pkl
OUTPUT_DIR = Path('../results/cv/figures')         # where to save plots/CSVs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Expert session criteria (must match what the cluster job used)
STAGE = 'Full_Task_Cont'
DISTRIBUTION = 'Uniform'
EXPERT_MIN_ACCURACY = 0.70
EXPERT_LAST_FRACTION = 0.50
MIN_EXPERT_SESSIONS = 5

## 2. Load Cluster Results

In [ ]:
all_results = load_cv_pickles(CV_RESULTS_DIR)
summarise_loaded_results(all_results)

In [ ]:
long_df = build_long_df(all_results)
print(f"Long DataFrame: {len(long_df)} rows")
print()
long_df.groupby(['animal_id', 'model'])['avg_test_error'].agg(
    ['mean', 'std', 'count']
)

## 3. ANOVA Model Comparison

For each animal, compare the distribution of 64 test errors between BE and SC
using one-way ANOVA. If p < 0.05, the model with lower mean error wins.

In [ ]:
comparison_df = run_anova(long_df)
comparison_df.style.format({
    'be_mean': '{:.5f}',
    'sc_mean': '{:.5f}',
    'F_stat': '{:.2f}',
    'p_value': '{:.2e}',
})

## 4. Per-Animal Violin + Paired Scatter

In [ ]:
for aid in sorted(long_df['animal_id'].unique()):
    fig = plot_cv_comparison(
        long_df, comparison_df, aid,
        fit_target='UM', output_dir=str(OUTPUT_DIR),
    )
    plt.show()
    plt.close(fig)

## 5. Winner Summary

In [ ]:
fig = plot_winner_summary(comparison_df, output_dir=str(OUTPUT_DIR))
plt.show()

## 6. Best-Fit Parameters

For each animal, extract the winning model's best-fit parameters
(from the seed with lowest avg test error).

In [ ]:
summary_df = build_summary_table(all_results, comparison_df)
summary_df

In [ ]:
# Print formatted parameter report
for _, row in summary_df.iterrows():
    aid = row['animal_id']
    winner = row['winner']
    p = row['p_value']
    print(f"\n{aid}: winner = {winner} (p = {p:.2e})")
    print(f"  BE mean error = {row['be_mean_error']:.5f}")
    print(f"  SC mean error = {row['sc_mean_error']:.5f}")
    if winner in ('BE', 'SC'):
        # Print the best-fit params
        param_cols = [c for c in summary_df.columns if c.startswith('best_') and c != 'best_seed']
        print(f"  Best seed = {row.get('best_seed', '?')}")
        for col in param_cols:
            val = row.get(col, np.nan)
            if pd.notna(val):
                name = col.replace('best_', '')
                print(f"  {name} = {val:.4f}")

## 7. Empirical vs Model Update Matrices

For each animal, simulate the winning model with its best-fit parameters
on the full expert dataset, and compare update matrices.

**Note**: This requires loading the original behavioural data via `behav_utils`.

In [ ]:
experiment = load_experiment(CONFIG_PATH)

# Build the flat DataFrames we need for update matrix computation
animal_dfs = {}
for aid in sorted(all_results.keys()):
    try:
        animal = experiment.get_animal(aid)
        expert = select_expert_sessions(
            animal, STAGE, DISTRIBUTION,
            EXPERT_MIN_ACCURACY, EXPERT_LAST_FRACTION,
        )
        animal_dfs[aid] = sessions_to_old_df(expert, animal_id=aid)
        print(f"  {aid}: {len(animal_dfs[aid])} trials, "
              f"{animal_dfs[aid]['block'].nunique()} blocks")
    except Exception as e:
        print(f"  {aid}: SKIP — {e}")

In [ ]:
for aid in sorted(animal_dfs.keys()):
    df = animal_dfs[aid]
    results_list = all_results[aid]
    winner_row = comparison_df[comparison_df['animal_id'] == aid]

    # Empirical UM
    emp_um, emp_cm = compute_empirical_um(df)

    # Model UMs
    model_ums = {}
    model_errors = {}
    for model_name in ['BE', 'SC']:
        params, seed = get_best_seed_params(results_list, model_name)
        if params is not None:
            try:
                m_um, _ = simulate_model_um(df, model_name, params, seed)
                # Reverse rows to match empirical convention
                m_um_rev = m_um[::-1]
                model_ums[model_name] = m_um_rev
                model_errors[model_name] = compute_matrix_error(m_um, emp_um[::-1])

                named = format_params(model_name, params)
                param_str = ', '.join(f'{k}={v:.3f}' for k, v in named.items())
                print(f"  {aid} {model_name}: {param_str}")
            except Exception as e:
                print(f"  {aid} {model_name}: simulation failed — {e}")

    if model_ums:
        fig = plot_um_comparison(
            emp_um, model_ums, model_errors, aid,
            output_dir=str(OUTPUT_DIR),
        )
        plt.show()
        plt.close(fig)

## 8. Parameter Distributions Across Seeds

In [ ]:
param_df = extract_param_df(all_results)
print(f"Parameter DataFrame: {len(param_df)} rows")
param_df.head(10)

In [ ]:
for aid in sorted(all_results.keys()):
    fig = plot_param_distributions(
        param_df, aid, output_dir=str(OUTPUT_DIR),
    )
    plt.show()
    plt.close(fig)

## 9. Save Outputs

In [ ]:
long_df.to_csv(OUTPUT_DIR / 'cv_test_errors_long.csv', index=False)
comparison_df.to_csv(OUTPUT_DIR / 'cv_comparison_anova.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'cv_summary.csv', index=False)
param_df.to_csv(OUTPUT_DIR / 'cv_best_params.csv', index=False)

print("Saved:")
for f in ['cv_test_errors_long.csv', 'cv_comparison_anova.csv',
          'cv_summary.csv', 'cv_best_params.csv']:
    print(f"  {OUTPUT_DIR / f}")

## Notes

**What this notebook does NOT do**:
- Run the CV itself (that's `cluster/submit_cv.sh` → `cluster/run_cv_single.py`)
- Merge raw pickles into a combined file (that's `cluster/gather_cv_results.py`)

**Caveats**:
- Block structure: with ~5–10 expert sessions, the 2-fold split may produce uneven folds.
  Monitor the fraction of NaN columns in the update matrix.
- Burn-in: the simulated 1000-trial burn-in assumes Uniform distribution. Correct for
  expert-phase fitting but would need adjustment for post-shift sessions.
- Update matrix orientation: the legacy code reverses rows (`::-1`) so that the bottom
  row = most negative stimulus. `compute_empirical_um` handles this automatically.
- The `best_params` in old pickles stores per-fold params (a list). New pickles also
  store `best_params_single` (from the best fold). `get_best_seed_params` handles both.